# Regressão linear simples

**Objetivo:** ajustar uma reta de duas formas — na mão, pelas fórmulas fechadas de mínimos quadrados, e com o `LinearRegression` do scikit-learn — e confirmar que dão o mesmo resultado. Depois, ler o coeficiente e olhar os resíduos.

In [ ]:
# bibliotecas base
import numpy as np
import pandas as pd

# Plotly para os gráficos (interativos e leves no Colab)
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
pio.templates.default = "simple_white"

# paleta do curso (a mesma do site)
AZUL, VERMELHO, VERDE = "#3266ad", "#c0392b", "#1a7a4a"
TINTA, SUAVE = "#1c1e15", "#6b7050"

# reprodutibilidade: uma única semente para tudo que é aleatório
SEMENTE = 42
np.random.seed(SEMENTE)

## 1. Os dados

Usamos o conjunto **diabetes** do scikit-learn e, por enquanto, um único preditor: o índice de massa corporal (`bmi`), já padronizado. O alvo `y` é uma medida de progressão da doença um ano depois.

In [ ]:
from sklearn.datasets import load_diabetes

dados = load_diabetes(as_frame=True)
x = dados.data["bmi"].values     # um preditor
y = dados.target.values          # alvo continuo
print("n =", len(x), "exemplos")
print("x[:5] =", np.round(x[:5], 3))
print("y[:5] =", y[:5])

## 2. Ajuste na mão

As fórmulas fechadas: a inclinação é a covariância de $x$ e $y$ dividida pela variância de $x$, e o intercepto ancora a reta no ponto médio.

$$\theta_1 = \frac{\sum_i (x_i-\bar{x})(y_i-\bar{y})}{\sum_i (x_i-\bar{x})^2}, \qquad \theta_0 = \bar{y} - \theta_1\bar{x}$$

In [ ]:
media_x = x.mean()
media_y = y.mean()
covariancia = ((x - media_x) * (y - media_y)).sum()
variancia_x = ((x - media_x) ** 2).sum()
theta1 = covariancia / variancia_x
theta0 = media_y - theta1 * media_x
print("theta1 (inclinacao):", round(theta1, 3))
print("theta0 (intercepto):", round(theta0, 3))

## 3. Ajuste com o scikit-learn

A mesma conta, com a interface `fit`. O scikit-learn espera `X` em formato de matriz `(n, p)`, então damos ao vetor `x` uma segunda dimensão.

In [ ]:
from sklearn.linear_model import LinearRegression

X = x.reshape(-1, 1)                 # de (n,) para (n, 1)
modelo = LinearRegression()
modelo.fit(X, y)
print("inclinacao sklearn:", round(modelo.coef_[0], 3))
print("intercepto sklearn:", round(modelo.intercept_, 3))
print("bate com a conta na mao?",
      np.allclose([modelo.coef_[0], modelo.intercept_], [theta1, theta0]))

## 4. A reta sobre os dados

Os pontos e a reta ajustada. Repare que a reta passa pelo centro da nuvem.

In [ ]:
grade_x = np.linspace(x.min(), x.max(), 100)
reta_y = theta0 + theta1 * grade_x

figura = go.Figure()
figura.add_trace(go.Scatter(x=x, y=y, mode="markers",
                            marker=dict(color=SUAVE, size=6, opacity=0.6), name="dados"))
figura.add_trace(go.Scatter(x=grade_x, y=reta_y, mode="lines",
                            line=dict(color=AZUL, width=3), name="reta ajustada"))
figura.update_layout(title="Regressao linear simples: bmi -> progressao",
                     xaxis_title="bmi (padronizado)", yaxis_title="progressao (y)",
                     height=380, margin=dict(l=10, r=10, t=50, b=10))
figura.show()

## 5. Resíduos

O resíduo é a distância vertical entre o ponto e a reta. Um gráfico de resíduos contra a predição deve parecer uma faixa **sem padrão** em torno do zero; qualquer curvatura sistemática seria sinal de que a relação não é bem uma reta.

In [ ]:
previsto = modelo.predict(X)
residuos = y - previsto

figura = go.Figure(go.Scatter(x=previsto, y=residuos, mode="markers",
                              marker=dict(color=VERMELHO, size=6, opacity=0.6)))
figura.add_hline(y=0, line_dash="dash", line_color=TINTA)
figura.update_layout(title="Residuos vs predicao",
                     xaxis_title="valor previsto", yaxis_title="residuo (y - y_previsto)",
                     height=340, margin=dict(l=10, r=10, t=50, b=10))
figura.show()
print("media dos residuos (deve ser ~0):", round(residuos.mean(), 6))

## Exercício

O `bmi` está padronizado, então o intercepto é o `y` previsto no `bmi` médio. Reajuste usando outro preditor (por exemplo `bp`, pressão) e compare a inclinação. Qual preditor sozinho explica melhor o alvo?

In [ ]:
# @title Solução (clique para revelar)
for nome in ["bmi", "bp", "s5"]:
    xi = dados.data[nome].values.reshape(-1, 1)
    m = LinearRegression().fit(xi, y)
    r2 = m.score(xi, y)
    print(nome, "-> inclinacao", round(m.coef_[0], 1), "| R2 =", round(r2, 3))
# o maior R2 indica o preditor que, sozinho, mais explica a variacao de y.